# Facial Visual Profile — RQ5
## Nonlinear fusion and auxiliary landmark supervision



In [1]:
# 1 — GPU, Drive and pinned source. No package upgrades.
import os, sys, json, hashlib, shutil, subprocess, platform, time, gc
from pathlib import Path
import importlib.metadata
import torch

assert torch.cuda.is_available(), (
    'CUDA 不可用'
    ' GPU 运行时仍显示 +cpu'
)
import torchvision
import numpy as np
import pandas as pd
from PIL import Image, __version__ as pillow_version
from google.colab import drive

drive.mount('/content/drive')
COMMIT = '1788617b6271dbb1d9b075660c0398cd2782eaf5'
REPO = Path('/content/fvp_rq5_notebook_repo')
PROJECT = Path('/content/drive/MyDrive/FacialVisualProfile')
STUDY = PROJECT/'runs/rq5_training_v1'
LOCAL = Path('/content/fvp_rq5_data')
IMAGES = LOCAL/'img_align_celeba'
ROOTS = [PROJECT/'runs/benchmark/day19/shards', PROJECT/'runs/benchmark/day20/shards']
assert PROJECT.is_dir()
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/DeweyYip/facial-visual-profile.git',str(REPO)],check=True)
def git(*args):
    return subprocess.check_output(['git','-C',str(REPO),*args],text=True).strip()
assert not git('status','--porcelain'), 'Pinned checkout changed; preserve changes and use a fresh runtime.'
if git('rev-parse','HEAD') != COMMIT:
    subprocess.run(['git','-C',str(REPO),'fetch','origin'],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','--detach',COMMIT],check=True)
assert git('rev-parse','HEAD') == COMMIT
STUDY.mkdir(parents=True,exist_ok=True)
LOCAL.mkdir(parents=True,exist_ok=True)
ENVIRONMENT = dict(python=platform.python_version(),torch=str(torch.__version__),
    torchvision=str(torchvision.__version__),numpy=np.__version__,Pillow=pillow_version,
    sklearn=importlib.metadata.version('scikit-learn'),cuda=torch.version.cuda,
    gpu=torch.cuda.get_device_name(0))
print(json.dumps(ENVIRONMENT,indent=2),flush=True)
print('SOURCE:',COMMIT,flush=True)
print('This run will train nine models after all preparation checks pass.',flush=True)


Mounted at /content/drive
{
  "python": "3.13.16",
  "torch": "2.11.0+cu130",
  "torchvision": "0.26.0+cu130",
  "numpy": "2.1.3",
  "Pillow": "11.3.0",
  "sklearn": "1.6.1",
  "cuda": "13.0",
  "gpu": "Tesla T4"
}
SOURCE: 1788617b6271dbb1d9b075660c0398cd2782eaf5
This run will train nine models after all preparation checks pass.


In [2]:
# 2 — Recorded runtime fixes, source snapshot and restart bindings.
import copy

def digest(path):
    h=hashlib.sha256()
    with open(path,'rb') as source:
        for block in iter(lambda:source.read(8*1024*1024),b''):
            h.update(block)
    return h.hexdigest()

def write_json(path,value):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.partial')
    temp.write_text(json.dumps(value,indent=2,allow_nan=False)+'\n')
    temp.replace(path)

def seal(path,value):
    if path.exists():
        assert json.loads(path.read_text())==value, f'Frozen record differs: {path}. Stop; do not delete old runs.'
    else:
        write_json(path,value)

# A separate local source copy keeps the pinned Git checkout pristine.
WORK=Path('/content/fvp_rq5_execution')
if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(REPO,WORK,ignore=shutil.ignore_patterns('.git','__pycache__'))
path=WORK/'training/rq5.py'
s=path.read_text()
def replace_once(old,new):
    global s
    assert s.count(old)==1, 'Pinned training source differs from reviewed code'
    s=s.replace(old,new,1)

# Completed runs must not rewrite best_model.pt, whose byte hash binds thresholds.
replace_once('    bind_run(folder,config,name,seed)\n', '''    bind_run(folder,config,name,seed)
    if (folder/'train_complete.json').exists():
        done = json.loads((folder/'train_complete.json').read_text())
        assert (folder/'latest.pt').exists() and (folder/'best_model.pt').exists()
        selected = torch.load(folder/'best_model.pt',map_location='cpu',weights_only=True)
        assert selected['exp']==name and selected['seed']==seed
        assert selected['epoch']==done['best']['epoch']
        print('SKIP completed training:',folder.name,'epochs:',done['epochs'],flush=True)
        return
    # A first-epoch disconnect can leave sidecars before latest.pt commits.
    # Preserve these uncommitted files, then repeat epoch 1 from its original seed.
    if not (folder/'latest.pt').exists():
        orphans = [folder/f for f in ('history.json','best.json','best_model.pt') if (folder/f).exists()]
        if orphans:
            import time
            recovery = folder/'uncommitted_recovery'/str(time.time_ns())
            recovery.mkdir(parents=True)
            for orphan in orphans:
                orphan.replace(recovery/orphan.name)
            print('Preserved uncommitted first-epoch sidecars:',recovery,flush=True)
''')
# Never change best checkpoint bytes when resuming a matching selected epoch.
replace_once("        write_torch(folder/'best_model.pt',dict(model=best_state,exp=name,seed=seed,epoch=best['epoch']))", '''        best_path = folder/'best_model.pt'
        existing = torch.load(best_path,map_location='cpu',weights_only=True) if best_path.exists() else None
        if existing is None or existing['epoch'] != best['epoch']:
            assert not (folder/'clean_val_thresholds.json').exists(), 'Do not replace a threshold-bound checkpoint'
            write_torch(best_path,dict(model=best_state,exp=name,seed=seed,epoch=best['epoch']))''')
replace_once("        history,best = saved['history'],saved['best']", "        history,best = saved['history'],saved['best']\n        print('RESUME:',folder.name,'last complete epoch:',saved['epoch'],flush=True)")
replace_once('        for batch in make_loader(train,config,True,seed+epoch):', '''        print('TRAIN:',folder.name,'epoch',epoch,'/',config['epochs'],flush=True)
        for batch_index,batch in enumerate(make_loader(train,config,True,seed+epoch)):
            if batch_index % 250 == 0:
                print('  train batches:',batch_index,'samples:',seen,'/',len(train),flush=True)''')
replace_once('    for batch in make_loader(dataset, config):', '''    for batch_index,batch in enumerate(make_loader(dataset, config)):
        if batch_index % 80 == 0:
            print('  inference batches:',batch_index,'/',len(dataset)//config['eval_batch_size']+1,flush=True)''')
# Binding must include NumPy/sklearn and device metadata as well as old fields.
replace_once("torchvision=torchvision.__version__,Pillow=pillow_version,cuda=torch.version.cuda)", "torchvision=torchvision.__version__,Pillow=pillow_version,cuda=torch.version.cuda,\n        numpy=np.__version__,sklearn=__import__('sklearn').__version__,gpu=torch.cuda.get_device_name(0))")
compile(s,str(path),'exec')
path.write_text(s)

os.chdir(WORK)
sys.path.insert(0,str(WORK))
# Drive keeps pretrained files too; no detector installation/inference is needed.
os.environ['TORCH_HOME']=str(PROJECT/'cache/rq5_torch_home')
os.environ['PYTHONUNBUFFERED']='1'
# Retain PyTorch defaults for TF32, matching the RQ4 code's lack of overrides.
from training import rq5 as train
from datasets.celeba_rq5 import CelebARQ5, ATTRIBUTES
from training.rq5_shared import set_seed

from models.rq5 import make_model
import datasets.celeba_rq5 as dataset_module
assert Path(train.__file__).resolve().is_relative_to(WORK), 'Restart runtime: stale training module import'
assert Path(dataset_module.__file__).resolve().is_relative_to(WORK), 'Restart runtime: local datasets package shadowed'
from scripts.evaluate_rq5 import conditions, report
from datasets.corruptions import apply_corruption, rgb_sha256
from training.rq5_shared import prepare_canvas, transform_corruption

# Pin auxiliary baseline source and the notebook patch, not just the Git commit.
SOURCE_FILES=['models/rq5.py','datasets/celeba_rq5.py','training/rq5.py',
    'training/rq5_shared.py','scripts/evaluate_rq5.py','datasets/landmark_cache.py',
    'datasets/corruptions.py','configs/rq5_training_v1.json']
SOURCE={p:digest(WORK/p) for p in SOURCE_FILES}
BINDING=dict(commit=COMMIT,notebook_revision='rq5_colab_v1',sources=SOURCE,
             environment=ENVIRONMENT,tf32_matmul=torch.backends.cuda.matmul.allow_tf32,
             tf32_cudnn=torch.backends.cudnn.allow_tf32)
seal(STUDY/'notebook_execution_binding.json',BINDING)
for relative in SOURCE_FILES:
    destination=STUDY/'source_snapshot'/relative
    destination.parent.mkdir(parents=True,exist_ok=True)
    if destination.exists():
        assert digest(destination)==SOURCE[relative]
    else:
        shutil.copyfile(WORK/relative,destination)
parameter_counts={}
for name in ('N0','N1','MTL'):
    temporary=make_model(name,pretrained=False)
    parameter_counts[name]=sum(p.numel() for p in temporary.parameters() if p.requires_grad)
    del temporary
assert parameter_counts=={'N0':11249368,'N1':11250776,'MTL':11193954}
seal(STUDY/'model_parameter_counts.json',parameter_counts)
print('Parameters:',parameter_counts,flush=True)
print('SOURCE / ENVIRONMENT BINDING PASS',flush=True)


Parameters: {'N0': 11249368, 'N1': 11250776, 'MTL': 11193954}
SOURCE / ENVIRONMENT BINDING PASS


In [3]:
# 3 — Original data integrity, annotation checks and resumable local extraction.
import zipfile
RAW=PROJECT/'data'
REQUIRED=['img_align_celeba.zip','list_eval_partition.txt','list_attr_celeba.txt','list_landmarks_align_celeba.txt']
for name in REQUIRED:
    assert (RAW/name).is_file(), f'Missing: {RAW/name}'
assert (PROJECT/'cache/mtcnn_clean.sqlite').is_file()
for root in ROOTS:
    assert root.is_dir(),f'Missing benchmark shards: {root}'

print('Hashing original CelebA ZIP and annotations...',flush=True)
EXPECTED_ZIP='46fb89443c578308acf364d7d379fe1b9efb793042c0af734b6112e4fd3a8c74'
assert digest(RAW/'img_align_celeba.zip')==EXPECTED_ZIP,'Original ZIP differs'
ASSETS={name:digest(RAW/name) for name in REQUIRED}
ASSETS['mtcnn_clean.sqlite']=digest(PROJECT/'cache/mtcnn_clean.sqlite')
seal(STUDY/'input_asset_binding.json',ASSETS)

pairs=[row.split() for row in (RAW/'list_eval_partition.txt').read_text(encoding='utf-8-sig').splitlines()]
partitions=dict(pairs)
assert len(pairs)==len(partitions)==202599
from collections import Counter
assert Counter(partitions.values())=={'0':162770,'1':19867,'2':19962}
# Same official ordering as RQ4, not a newly shuffled/sampled split.
for split in ('0','1','2'):
    selected=[name for name,code in pairs if code==split]
    assert selected==sorted(selected),'RQ4 partition file is not filename-sorted; do not silently change order'

archive=LOCAL/'img_align_celeba.zip'
if not archive.exists() or digest(archive)!=EXPECTED_ZIP:
    print('Copying ZIP from Drive to local disk...',flush=True)
    temp=archive.with_name(archive.name+'.partial')
    shutil.copyfile(RAW/'img_align_celeba.zip',temp)
    assert digest(temp)==EXPECTED_ZIP
    temp.replace(archive)
IMAGES.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(archive) as z:
    members={}
    for info in z.infolist():
        if info.is_dir():
            continue
        name=Path(info.filename).name
        assert name not in members and name in partitions
        members[name]=info
    assert set(members)==set(partitions)
    for i,name in enumerate(sorted(members),1):
        info=members[name]; dst=IMAGES/name
        if not dst.is_file() or dst.stat().st_size!=info.file_size:
            temp=dst.with_name(dst.name+'.partial')
            with z.open(info) as source, temp.open('wb') as output:
                shutil.copyfileobj(source,output)
            temp.replace(dst)
        if i%40000==0:
            print('Local images ready:',i,'/',len(members),flush=True)
assert all((IMAGES/name).is_file() for name in partitions)
print('DATA PREPARATION PASS:',len(partitions),'images',flush=True)


Hashing original CelebA ZIP and annotations...
Copying ZIP from Drive to local disk...
Local images ready: 40000 / 202599
Local images ready: 80000 / 202599
Local images ready: 120000 / 202599
Local images ready: 160000 / 202599
Local images ready: 200000 / 202599
DATA PREPARATION PASS: 202599 images


In [4]:
# 4 — Audit all three original RQ4-A runs before adding MTL.
BASELINE=PROJECT/'runs/rq4_training_v1'
old_protocol=json.loads((BASELINE/'study_protocol.json').read_text())
expected=dict(epochs=10,batch_size=64,lr=1e-4,weight_decay=0.,num_workers=2,
    image_size=224,pretrained='ResNet18_Weights.IMAGENET1K_V1',
    checkpoint_selection='maximum clean-validation mAP; earliest epoch on exact tie')
for key,value in expected.items():
    assert old_protocol[key]==value, f'RQ4 protocol differs: {key}'
assert old_protocol['attributes']==ATTRIBUTES
assert old_protocol['seeds']==[0,1,2]
assert old_protocol['experiments']['A']==dict(geometry=False,augmentation=False,mask_rgb=False)
policy=json.loads((BASELINE/'early_stopping_policy.json').read_text())
assert policy['enabled'] is True
assert (policy['max_epochs'],policy['min_epochs'],policy['patience'])==(10,4,2)
old_runs={}
for seed in (0,1,2):
    folder=BASELINE/f'A_seed{seed}'
    for filename in ['config.json','history.json','best.json','best_model.pt',
                     'train_complete.json','clean_val_thresholds.json','test_metrics.json','test_complete.json']:
        assert (folder/filename).is_file(),f'Missing old baseline: {folder/filename}'
    cfg=json.loads((folder/'config.json').read_text())
    assert cfg['exp']=='A' and cfg['seed']==seed
    assert cfg['protocol']==old_protocol and cfg['options']==old_protocol['experiments']['A']
    thresholds=json.loads((folder/'clean_val_thresholds.json').read_text())
    assert thresholds['attribute_order']==ATTRIBUTES and len(thresholds['thresholds'])==24
    results=json.loads((folder/'test_metrics.json').read_text())
    assert set(results)=={label for label,_,_ in conditions()}
    history=json.loads((folder/'history.json').read_text())
    best=json.loads((folder/'best.json').read_text())
    selected=max(history,key=lambda row:row['clean_validation_mAP'])
    assert selected['epoch']==best['epoch']
    done=json.loads((folder/'train_complete.json').read_text())
    assert done['epochs']==history[-1]['epoch']
    old_runs[str(seed)]={name:digest(folder/name) for name in
        ['config.json','history.json','best.json','best_model.pt','clean_val_thresholds.json','test_metrics.json']}
    print('RQ4 A seed',seed,'audit PASS; selected epoch',best['epoch'],flush=True)
seal(STUDY/'rq4_baseline_binding.json',old_runs)
if (BASELINE/'runtime_latest.json').exists():
    runtime=json.loads((BASELINE/'runtime_latest.json').read_text())
    print('Historical RQ4 latest runtime record (not per-seed proof):',runtime,flush=True)
    print('Current runtime:',ENVIRONMENT,flush=True)
    write_json(STUDY/'historical_environment_comparison.json',dict(
        old_latest_record=runtime,current=ENVIRONMENT,
        interpretation='Environment differences limit strict paired causal interpretation; no A retraining.'))
print('RQ4-A protocol/records audit PASS. Historical metric bytes remain untouched.',flush=True)


RQ4 A seed 0 audit PASS; selected epoch 2
RQ4 A seed 1 audit PASS; selected epoch 2
RQ4 A seed 2 audit PASS; selected epoch 2
Historical RQ4 latest runtime record (not per-seed proof): {'python': '3.13.15', 'torch': '2.11.0+cu130', 'torchvision': '0.26.0+cu130', 'Pillow': '11.3.0', 'device': 'cuda', 'cuda': '13.0'}
Current runtime: {'python': '3.13.16', 'torch': '2.11.0+cu130', 'torchvision': '0.26.0+cu130', 'numpy': '2.1.3', 'Pillow': '11.3.0', 'sklearn': '1.6.1', 'cuda': '13.0', 'gpu': 'Tesla T4'}
RQ4-A protocol/records audit PASS. Historical metric bytes remain untouched.


In [5]:
# 5 — Reuse archived preflight and frozen lambda.
saved_binding = json.loads(
    (STUDY / "notebook_execution_binding.json").read_text()
)
assert saved_binding == BINDING, "源码或环境与原运行不同，停止。"

saved_assets = json.loads(
    (STUDY / "input_asset_binding.json").read_text()
)
assert saved_assets == ASSETS, "输入资产与原运行不同，停止。"

preflight = json.loads(
    (STUDY / "frozen_input_preflight.json").read_text()
)
expected = {label for label, _, _ in conditions()}
assert len(preflight) == 21
assert {row["condition"] for row in preflight} == expected

scale = json.loads(
    (STUDY / "lambda_scale_check.json").read_text()
)
frozen = json.loads(
    (STUDY / "frozen_config.json").read_text()
)
assert scale["source"] == SOURCE
assert frozen["mtl_lambda"] == scale["lambda"]

print("SKIP: 已完成的 21 条件检查。")
print("已冻结 λ =", frozen["mtl_lambda"])

Inspecting original-image caches...
train cache statuses: {'ok': 162521, 'no_face': 249}
  train ok 000001.jpg PASS
  train no_face 000199.jpg PASS
valid cache statuses: {'ok': 19835, 'no_face': 32}
  valid ok 162771.jpg PASS
  valid no_face 162871.jpg PASS
Frozen input/cache parity: clean PASS
Frozen input/cache parity: blur_s1 PASS
Frozen input/cache parity: blur_s2 PASS
Frozen input/cache parity: blur_s3 PASS
Frozen input/cache parity: blur_s4 PASS
Frozen input/cache parity: brightness_s1 PASS
Frozen input/cache parity: brightness_s2 PASS
Frozen input/cache parity: brightness_s3 PASS
Frozen input/cache parity: brightness_s4 PASS
Frozen input/cache parity: rotation_s1 PASS
Frozen input/cache parity: rotation_s2 PASS
Frozen input/cache parity: rotation_s3 PASS
Frozen input/cache parity: rotation_s4 PASS
Frozen input/cache parity: occlusion_s1 PASS
Frozen input/cache parity: occlusion_s2 PASS
Frozen input/cache parity: occlusion_s3 PASS
Frozen input/cache parity: occlusion_s4 PASS
Froz

## settle λ, and then train




In [6]:
# 6 — Freeze one training-only loss-scale lambda. No trial training.
CONFIG_PATH=STUDY/'frozen_config.json'
SCALE_PATH=STUDY/'lambda_scale_check.json'
base_config=train.load_config(WORK/'configs/rq5_training_v1.json')
if SCALE_PATH.exists():
    scale=json.loads(SCALE_PATH.read_text())
    assert scale['rule']=='mean_BCE_over_mean_MSE_4_train_batches_6_significant_digits_v1'
    print('REUSE frozen lambda:',scale['lambda'],flush=True)
else:
    # Never derive a new lambda once any official run has started.
    assert not any((STUDY/f'{name}_seed{seed}'/'config.json').exists()
                   for name in base_config['models'] for seed in base_config['seeds'])
    set_seed(0)
    model=make_model('MTL',pretrained=True).cuda().train()
    ds=CelebARQ5(PROJECT,IMAGES,'train','MTL')
    values=[]
    with torch.no_grad():
        for i,batch in enumerate(train.make_loader(ds,base_config,True,seed=1)):
            logits,coordinates=model(batch['image'].cuda())
            bce=torch.nn.functional.binary_cross_entropy_with_logits(logits,batch['target'].cuda())
            mse=torch.nn.functional.mse_loss(coordinates,batch['gt'].cuda())
            assert torch.isfinite(bce) and torch.isfinite(mse) and mse>0
            values.append(dict(batch=i,BCE=bce.item(),MSE=mse.item(),names=batch['name']))
            if i==3:
                break
    weight=float(f"{np.mean([v['BCE'] for v in values])/np.mean([v['MSE'] for v in values]):.6g}")
    assert np.isfinite(weight) and weight>0
    scale=dict(rule='mean_BCE_over_mean_MSE_4_train_batches_6_significant_digits_v1',
        seed=0,loader_seed=1,lambda_value=weight,batches=values,source=SOURCE)
    scale['lambda']=weight
    write_json(SCALE_PATH,scale)
    del model,ds,batch
    gc.collect();torch.cuda.empty_cache()
CONFIG=dict(base_config)
CONFIG['mtl_lambda']=scale['lambda']
CONFIG['lambda_basis']=scale['rule']+'; see lambda_scale_check.json'
seal(CONFIG_PATH,CONFIG)
assert train.load_config(CONFIG_PATH)==CONFIG
print('FROZEN lambda:',CONFIG['mtl_lambda'],flush=True)
print('Config:',CONFIG_PATH,flush=True)


REUSE frozen lambda: 1.89811
FROZEN lambda: 1.89811
Config: /content/drive/MyDrive/FacialVisualProfile/runs/rq5_training_v1/frozen_config.json


In [7]:
# 7 — Run all nine trainings and threshold exports with logs and epoch resume.
# Logs append across sessions; a subprocess failure stops this notebook.
def logged(command,log_path):
    log_path.parent.mkdir(parents=True,exist_ok=True)
    with log_path.open('a',buffering=1) as log:
        log.write('\nSESSION '+time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())+'\n')
        process=subprocess.Popen(command,cwd=str(WORK),stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1,env=os.environ.copy())
        try:
            for line in process.stdout:
                print(line,end='',flush=True)
                log.write(line)
            status=process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill();process.wait()
            raise
        assert status==0,f'Execution stopped ({status}); inspect {log_path}. Resume by rerunning notebook.'

for name in CONFIG['models']:
    for seed in CONFIG['seeds']:
        print('\n=== TRAIN / RESUME:',name,'seed',seed,'===',flush=True)
        logged([sys.executable,'-u','-m','training.rq5',
            '--project',str(PROJECT),'--images',str(IMAGES),'--config',str(CONFIG_PATH),
            '--model',name,'--seed',str(seed)],STUDY/'logs'/f'{name}_seed{seed}_training.log')

for name in CONFIG['models']:
    for seed in CONFIG['seeds']:
        folder=STUDY/f'{name}_seed{seed}'
        assert (folder/'train_complete.json').is_file()
        assert (folder/'clean_val_thresholds.json').is_file()
print('ALL NINE TRAININGS AND FROZEN THRESHOLDS COMPLETE',flush=True)



=== TRAIN / RESUME: N0 seed 0 ===
RESUME: N0_seed0 last complete epoch: 2
TRAIN: N0_seed0 epoch 3 / 10
  train batches: 0 samples: 0 / 162770
  train batches: 250 samples: 16000 / 162770
  train batches: 500 samples: 32000 / 162770
  train batches: 750 samples: 48000 / 162770
  train batches: 1000 samples: 64000 / 162770
  train batches: 1250 samples: 80000 / 162770
  train batches: 1500 samples: 96000 / 162770
  train batches: 1750 samples: 112000 / 162770
  train batches: 2000 samples: 128000 / 162770
  train batches: 2250 samples: 144000 / 162770
  train batches: 2500 samples: 160000 / 162770
  inference batches: 0 / 156
  inference batches: 80 / 156
N0 0 {'epoch': 3, 'training_loss': 0.17607025218356365, 'attribute_BCE': 0.17607025218356365, 'landmark_MSE': 0.0, 'clean_validation_mAP': 0.8165720348143997, 'clean_validation_F1_at_05': 0.7275355406994155}
TRAIN: N0_seed0 epoch 4 / 10
  train batches: 0 samples: 0 / 162770
  train batches: 250 samples: 16000 / 162770
  train batches:

In [8]:
# 8 — Final 21-condition evaluation. Resume at condition boundaries.
# N1 first: all samples are checked against archived final RGB hashes.
for name in ('N1','N0','MTL'):
    for seed in CONFIG['seeds']:
        print('\n=== FINAL TEST / RESUME:',name,'seed',seed,'===',flush=True)
        command=[sys.executable,'-u','scripts/evaluate_rq5.py',
            '--project',str(PROJECT),'--images',str(IMAGES),'--config',str(CONFIG_PATH),
            '--model',name,'--seed',str(seed)]
        if name=='N1':
            command+=['--benchmark-roots',*[str(root) for root in ROOTS]]
        logged(command,STUDY/'logs'/f'{name}_seed{seed}_test.log')
print('ALL NINE 21-CONDITION EVALUATIONS COMPLETE',flush=True)



=== FINAL TEST / RESUME: N1 seed 0 ===
  inference batches: 0 / 156
  inference batches: 80 / 156
N1 0 clean 0.8202498088010248 0.7581471534728657
  inference batches: 0 / 156
  inference batches: 80 / 156
N1 0 blur_s1 0.8055081573314311 0.7107474305246607
  inference batches: 0 / 156
  inference batches: 80 / 156
N1 0 blur_s2 0.7510074526419223 0.5298718805910845
  inference batches: 0 / 156
  inference batches: 80 / 156
N1 0 blur_s3 0.6683483637967353 0.2849539513279555
  inference batches: 0 / 156
  inference batches: 80 / 156
N1 0 blur_s4 0.5619453066566931 0.13637323737902546
  inference batches: 0 / 156
  inference batches: 80 / 156
N1 0 brightness_s1 0.8192157312684186 0.7542625047029151
  inference batches: 0 / 156
  inference batches: 80 / 156
N1 0 brightness_s2 0.8143373689425285 0.7429876552856163
  inference batches: 0 / 156
  inference batches: 80 / 156
N1 0 brightness_s3 0.8013652209320044 0.7099655675963525
  inference batches: 0 / 156
  inference batches: 80 / 156
N1 0

In [9]:
# 9 — Complete reports, paired effects, sample SD and immutable baseline check.
for seed in (0,1,2):
    folder=BASELINE/f'A_seed{seed}'
    assert {name:digest(folder/name) for name in old_runs[str(seed)]}==old_runs[str(seed)],'Historical A changed'
report(PROJECT,CONFIG)
print('Corruption averages (clean excluded):')
display(pd.read_csv(STUDY/'corruption_summary.csv',header=[0,1],index_col=0))
print('Paired seed contrasts in percentage points:')
display(pd.read_csv(STUDY/'paired_seed_contrasts.csv'))
status={}
for name in CONFIG['models']:
    for seed in CONFIG['seeds']:
        folder=STUDY/f'{name}_seed{seed}'
        status[f'{name}_seed{seed}']=dict(
            training=json.loads((folder/'train_complete.json').read_text()),
            checkpoint_sha256=digest(folder/'best_model.pt'),
            thresholds_sha256=digest(folder/'clean_val_thresholds.json'),
            test_sha256=digest(folder/'test_metrics.json'))
write_json(STUDY/'rq5_complete.json',dict(commit=COMMIT,execution_binding=BINDING,
    lambda_value=CONFIG['mtl_lambda'],runs=status,
    note='Three seeds; sample SD is descriptive. Negative findings must be retained.'))
print('RQ5 COMPLETE:',STUDY,flush=True)
print('CSV outputs remain in the new RQ5 folder; old RQ4/Oracle artifacts were not modified.')


Corruption averages (clean excluded):


mAP           frozen_F1            F1_at_05          
           mean       std      mean       std      mean       std
model                                                            
A      0.760485  0.002264  0.613690  0.000473  0.564882  0.007329
MTL    0.755648  0.005447  0.618356  0.006477  0.570936  0.012575
N0     0.760157  0.001216  0.613751  0.004459  0.563436  0.007827
N1     0.760220  0.002524  0.610896  0.000920  0.562335  0.009570

Paired seed contrasts in percentage points:


,comparison,scope,seed,delta_mAP_pp,delta_frozen_F1_pp,delta_F1_at_05_pp
0,N1-N0,clean,0,0.042985,0.152076,0.320631
1,N1-N0,clean,1,-0.036899,-0.252878,-0.147007
2,N1-N0,clean,2,0.021616,-0.241581,0.076732
3,MTL-A,clean,0,0.014879,0.104065,-1.461406
4,MTL-A,clean,1,-0.070391,0.379333,0.237446
5,MTL-A,clean,2,-0.041883,0.002231,1.456162
6,N0-A,clean,0,-0.009928,-0.000925,-0.320839
7,N0-A,clean,1,-0.061052,0.114764,-0.663546
8,N0-A,clean,2,-0.240816,-0.029410,-0.047624
9,N1-N0,corrupted,0,0.030122,-0.335327,0.109636


RQ5 COMPLETE: /content/drive/MyDrive/FacialVisualProfile/runs/rq5_training_v1
CSV outputs remain in the new RQ5 folder; old RQ4/Oracle artifacts were not modified.
